In [3]:
import os
import sys
import gc
os.chdir('/content')

print("[DeFooocus] Preparing ...")

theme = "dark" #@param ["dark", "light"]
preset = "sd1.5" #@param ["deafult", "realistic", "anime", "lcm", "sai", "turbo", "lighting", "hypersd", "playground_v2.5", "dpo", "spo", "sd1.5"]
advenced_args = "--share --attention-split --always-low-vram" #@param {type: "string"}

if preset != "deafult":
  args = f"{advenced_args} --theme {theme} --preset {preset}"
elif preset == "deafult":
  args = f"{advenced_args} --theme {theme}"

# Clean up previous clone completely
!rm -rf /content/DeFooocus
!git clone https://github.com/ehristoforu/DeFooocus.git
%cd /content/DeFooocus

# Upgrade pip first to ensure better wheel matching on Python 3.13
!pip install --upgrade pip

# Safely strip all references of pygit2 to fully bypass pip source compiling
!sed -i '/pygit2/d' requirements_versions.txt

# Modify requirements_versions.txt to remove strict version constraints that trigger source compilation
!sed -i 's/onnxruntime==1.18.1/onnxruntime/g' requirements_versions.txt
!sed -i 's/rembg==2.0.57/rembg/g' requirements_versions.txt
!sed -i '/gradio/d' requirements_versions.txt
!sed -i '/safetensors/d' requirements_versions.txt
!sed -i '/tokenizers/d' requirements_versions.txt
!sed -i '/transformers/d' requirements_versions.txt
!sed -i '/numpy/d' requirements_versions.txt
!sed -i '/scipy/d' requirements_versions.txt
!sed -i '/pyyaml/d' requirements_versions.txt
!sed -i '/pillow/d' requirements_versions.txt
!sed -i '/accelerate/d' requirements_versions.txt

# Install modern compatible versions first via binary wheels
!pip install --no-cache-dir --prefer-binary --upgrade "numpy>=2.2.0" torchsde tokenizers transformers scipy PyYAML pillow accelerate

# Force install Gradio 3.50.2 and its dependencies bypassing constraints
!pip install --no-cache-dir --no-deps "gradio==3.50.2" "gradio-client==0.6.1"
!pip install --no-cache-dir ffmpy "importlib-resources<7.0,>=1.3"

# Uninstall einops to resolve version conflict before installing requirements
!pip uninstall -y einops

# Install remaining requirements preferring binary wheels while maintaining compatibility
!pip install --prefer-binary --no-cache-dir --upgrade -r requirements_versions.txt

# Apply a comprehensive hotfix directly to Starlette and Jinja2 template handlers to resolve dict argument errors
with open('modules/ui_gradio_extensions.py', 'r') as f:
    ui_content = f.read()

jinja_starlette_hotfix = """import starlette.templating
import jinja2.utils
import gradio

# 0. Hotfix Gradio's internal package/bundling check so share mode is allowed
import gradio.utils
gradio.utils.is_bundled = lambda *args, **kwargs: True

# Also patch the actual exception-raising function inside gradio.routes if applicable
try:
    import gradio.routes
    gradio.routes.is_bundled = lambda *args, **kwargs: True
except Exception:
    pass

# 1. Hotfix Starlette's template resolution to ensure 'name' is always a string
original_get_template = starlette.templating.Jinja2Templates.get_template

def safe_get_template(self, name, *args, **kwargs):
    if isinstance(name, dict):
        # Extract default index if dict configuration is erroneously passed as name
        name = name.get('template', 'index.html')
    return original_get_template(self, str(name), *args, **kwargs)

starlette.templating.Jinja2Templates.get_template = safe_get_template

# 2. Hotfix Jinja2's LRUCache to completely ignore unhashable keys (like nested dicts) without raising exceptions
original_getitem = jinja2.utils.LRUCache.__getitem__
original_setitem = jinja2.utils.LRUCache.__setitem__
original_contains = jinja2.utils.LRUCache.__contains__
original_get = jinja2.utils.LRUCache.get

def safe_getitem(self, key):
    try:
        return original_getitem(self, key)
    except TypeError:
        return None

def safe_setitem(self, key, value):
    try:
        original_setitem(self, key, value)
    except TypeError:
        pass

def safe_contains(self, key):
    try:
        return original_contains(self, key)
    except TypeError:
        return False

def safe_get(self, key, default=None):
    try:
        return original_get(self, key, default)
    except TypeError:
        return default

jinja2.utils.LRUCache.__getitem__ = safe_getitem
jinja2.utils.LRUCache.__setitem__ = safe_setitem
jinja2.utils.LRUCache.__contains__ = safe_contains
jinja2.utils.LRUCache.get = safe_get

# 3. Patch Starlette's TemplateResponse to sanitize context dicts
original_template_response = starlette.templating.Jinja2Templates.TemplateResponse

def safe_template_response(self, name, context, *args, **kwargs):
    if isinstance(name, dict):
        name = name.get('template', 'index.html')
    sanitized_context = {}
    for k, v in context.items():
        if isinstance(v, dict):
            class HashableDict(dict):
                def __hash__(self):
                    return hash(frozenset((kk, str(vv)) for kk, vv in self.items()))
            sanitized_context[k] = HashableDict(v)
        else:
            sanitized_context[k] = v
    return original_template_response(self, str(name), sanitized_context, *args, **kwargs)

starlette.templating.Jinja2Templates.TemplateResponse = safe_template_response
"""

# Insert this global catch-all hotfix at the very top of ui_gradio_extensions.py
ui_content = jinja_starlette_hotfix + "\n" + ui_content

with open('modules/ui_gradio_extensions.py', 'w') as f:
    f.write(ui_content)

# Patch patch_clip.py to fix AttributeError: module 'transformers.modeling_utils' has no attribute 'no_init_weights'
with open('modules/patch_clip.py', 'r') as f:
    clip_content = f.read()

# Add the compatibility context manager fallback at the very beginning of the file
clip_patch = """import contextlib
import transformers.modeling_utils as modeling_utils
if not hasattr(modeling_utils, 'no_init_weights'):
    @contextlib.contextmanager
    def dummy_no_init_weights():
        yield
    modeling_utils.no_init_weights = dummy_no_init_weights
"""

# Inject the monkeypatch at the top of the file to make sure it executes first
clip_content = clip_patch + "\n" + clip_content

# Safely bypass the missing nested text_model attribute on newer transformers CLIPTextModel instances
clip_content = clip_content.replace(
    "self.transformer.text_model.embeddings.to(torch.float32)",
    "(self.transformer.text_model if hasattr(self.transformer, 'text_model') else self.transformer).embeddings.to(torch.float32)"
)

with open('modules/patch_clip.py', 'w') as f:
    f.write(clip_content)

# Hotfix CPU-only fallback inside model_management.py
with open('ldm_patched/modules/model_management.py', 'r') as f:
    mm_content = f.read()

# Inject a safe fallback wrapper on get_torch_device to prevent AssertionError if CUDA is missing
safe_device_patch = """
import torch
_orig_get_torch_device = get_torch_device
def get_torch_device():
    try:
        if not torch.cuda.is_available():
            return torch.device('cpu')
        return _orig_get_torch_device()
    except Exception:
        return torch.device('cpu')
"""

with open('ldm_patched/modules/model_management.py', 'w') as f:
    f.write(mm_content + "\n" + safe_device_patch)

# Clean system memory as much as possible before starting imports
gc.collect()

print(f"[DeFooocus] Starting clean subprocess with args: {args}")
!python launch.py {args}

[DeFooocus] Preparing ...
Cloning into 'DeFooocus'...
remote: Enumerating objects: 7032, done.
remote: Total 7032 (delta 0), reused 0 (delta 0), pack-reused 7032 (from 1)
Receiving objects: 100% (7032/7032), 47.35 MiB | 33.67 MiB/s, done.
Resolving deltas: 100% (3825/3825), done.
/content/DeFooocus
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 34.7 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 3.50.2 requires aiofiles<24.0,>=22.0, but you have aiofiles 25.1.0 which is incompatible.
gradio 3.50.2 requires markupsafe~=2.0, but you have markupsafe 3.0.3 which is incompatible.
gradio 3.50.2 requires numpy~=1.0, but you have numpy 2.5.3 which is incompatible.
gradio 3.50.2 requires pillo

Found existing installation: einops 0.8.0
Uninstalling einops-0.8.0:
  Successfully uninstalled einops-0.8.0
INFO: pip is looking at multiple versions of rembg to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of rembg to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 173.1 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.3
    Uninstalling numpy-2.5.3:
      Successfully uninstalled numpy-2.5.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [einops]
ERROR: pip's dependency resolver does not currently take into account all the

[DeFooocus] Starting clean subprocess with args: --share --attention-split --always-low-vram --theme dark --preset sd1.5
[System ARGV] ['launch.py', '--share', '--attention-split', '--always-low-vram', '--theme', 'dark', '--preset', 'sd1.5']
/content/DeFooocus/build_launcher.py:9: SyntaxWarning: invalid escape sequence '\p'
  .\python_embeded\python.exe -s DeFooocus\entry_with_update.py {cmds} %*
Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Fooocus version: 0.2
Loaded preset: /content/DeFooocus/presets/sd1.5.json
Downloading: "https://huggingface.co/lllyasviel/misc/resolve/main/xlvaeapp.pth" to /content/DeFooocus/models/vae_approx/xlvaeapp.pth

100% 209k/209k [00:00<00:00, 1.18MB/s]
Downloading: "https://huggingface.co/lllyasviel/misc/resolve/main/vaeapp_sd15.pt" to /content/DeFooocus/models/vae_approx/vaeapp_sd15.pth

100% 209k/209k [00:00<00:00, 1.84MB/s]
Downloading: "https://huggingface.co/lllyasviel/misc/resolve/main/xl-to-v1_interposer-v3.1.safetensors" to /content/D